# MLFF 학습 v2 (Kaggle GPU)

Colab GPU 할당량 소진에 대비한 Kaggle 버전입니다. 주 30시간 무료 GPU를 씁니다.

## Kaggle의 결정적 장점

Colab은 90분 방치하면 연결이 끊겨서 몇 시간짜리 학습을 지켜봐야 했습니다.
Kaggle은 **`Save Version` → `Save & Run All`** 로 제출하면
**브라우저를 꺼도 백그라운드에서 최대 12시간 실행**됩니다. 지켜볼 필요가 없습니다.

---

## 실행 전 준비 3가지

### 1) 데이터를 Kaggle Dataset으로 업로드 (최초 1회만)

1. [kaggle.com/datasets](https://www.kaggle.com/datasets) 접속 → 우측 상단 **`New Dataset`**
2. 맥북 `~/mlff-project/data/processed/` 에서 **`train.xyz`(107MB)** 와 **`valid.xyz`(26MB)** 를 끌어다 놓기
3. 제목을 **`mlff-sin`** 으로 입력 (다른 이름도 되지만 아래 셀이 자동으로 찾습니다)
4. **`Create`** 클릭 → 업로드 완료까지 몇 분 대기

### 2) 이 노트북에 데이터 연결

우측 패널 **`Input`** → **`+ Add Input`** → 방금 만든 데이터셋 검색 후 **`+`** 클릭

### 3) 우측 설정 패널에서 두 가지 켜기

| 설정 | 값 | 안 켜면 |
|---|---|---|
| **Accelerator** | `GPU T4 x2` 또는 `GPU P100` | CPU로 돌아 매우 느림 |
| **Internet** | `On` | 패키지 설치와 모델 다운로드 실패 |

> Internet을 켜려면 Kaggle 계정 휴대폰 인증이 필요할 수 있습니다.

---

준비가 끝나면 아래 셀을 순서대로 실행하거나, 바로 **`Save Version` → `Save & Run All`** 하세요.

## 1. GPU와 데이터 확인

In [ ]:
import os, glob, shutil, subprocess

# --- GPU 확인 ---
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
else:
    print('nvidia-smi 없음 -> 현재 CPU 런타임입니다')

import torch
if torch.cuda.is_available():
    print('CUDA 사용 가능: True')
else:
    print()
    print('=' * 60)
    print('GPU가 꺼져 있습니다. 아래대로 켜고 이 셀을 다시 실행하세요.')
    print('  우측 패널 -> Accelerator -> "GPU T4 x2" 또는 "GPU P100" 선택')
    print('  (우측 패널이 안 보이면 우측 상단 화살표나 톱니바퀴 아이콘 클릭)')
    print('  변경 시 세션이 재시작됩니다.')
    print('=' * 60)

# --- 데이터 자동 탐색 (폴더 이름을 몰라도 됨) ---
TRAIN_FILE = VALID_FILE = None
for path in glob.glob('/kaggle/input/**/*.xyz', recursive=True):
    name = os.path.basename(path)
    if name == 'train.xyz':
        TRAIN_FILE = path
    elif name == 'valid.xyz':
        VALID_FILE = path

print()
for label, path in [('train', TRAIN_FILE), ('valid', VALID_FILE)]:
    if path:
        print(f'OK   {label}: {path}  ({os.path.getsize(path)/1024/1024:.1f} MB)')
    else:
        print(f'없음 {label}.xyz  -> 우측 Input 패널에서 mlff-sin 데이터셋을 연결하세요')

## 2. 패키지 설치

Internet 설정이 켜져 있어야 합니다. 2~3분 걸립니다.

In [ ]:
!pip install -q mace-torch ase 2>&1 | tail -3

import mace, ase
print('설치 완료 / mace:', mace.__version__, '/ ase:', ase.__version__)

## 3. 데이터 구조 개수 확인

In [ ]:
from ase.io import read

n_train = len(read(TRAIN_FILE, ':'))
n_valid = len(read(VALID_FILE, ':'))
print('train 구조 수:', n_train, '(정상: 17992)')
print('valid 구조 수:', n_valid, '(정상: 4518)')

## 4. 학습 설정

### v1(Colab)에서 배운 것

`huber(delta=0.01) + lr=0.01 + forces_weight=100`으로 돌렸다가 실패했습니다.

| epoch | 에너지 | 힘 |
|---|---|---|
| 14 | 19.5 | **337** ← 최저 |
| 28 | 20.7 | **1188** ← 3.5배 발산 |

원인이 세 가지였습니다.

1. **`huber_delta=0.01`이 너무 작음** — 오차가 0.01을 넘으면 벌점이 포화되어 모델이 어려운 구조를 포기
2. **`forces_weight=100`은 논문 검증 범위 밖** — 논문(SAIT) 권장은 `1 ≤ λf < 10`
3. **안전장치가 전부 꺼져 있었음** — `patience=2048`(사실상 off), `scheduler_patience=50`(짧은 실행에선 미작동)

### v2 설정과 근거

| 설정 | v1 | v2 | 근거 |
|---|---|---|---|
| `LOSS` | huber | **weighted** | A/B 실험에서 우세 (344 vs 579) |
| `LR` | 0.01 | **0.005** | 0.01은 두 실험 모두에서 발산 |
| `FORCES_WEIGHT` | 100 | **1** | 논문 권장 구간의 하한 |
| `SCHEDULER_PATIENCE` | 50 | **3** | 3회 정체 시 학습률 20% 감소 |
| `PATIENCE` | 2048 | **10** | 10회 정체 시 중단 + 최고 모델 저장 |

### 자동 종료

```
정체 3회  -> 학습률 20% 감소 (회복 시도)
정체 6회  -> 학습률 추가 감소
정체 10회 -> 중단하고 최고 성적 모델 저장
```

In [ ]:
NAME = 'mace_kaggle_v2'

# --- 핵심 설정 ---
LR = 0.005
LOSS = 'weighted'
FORCES_WEIGHT = 1         # 논문 권장 1 <= lf < 10 의 하한. MACE 기본값 100은 검증 범위 밖
ENERGY_WEIGHT = 1
SCHEDULER_PATIENCE = 3
PATIENCE = 10
EPOCHS = 40               # 상한. 실제 종료는 PATIENCE가 결정

BATCH_SIZE = 16
DTYPE = 'float32'
HUBER_DELTA = 0.01        # LOSS='huber'일 때만 사용

# Kaggle은 /kaggle/working/ 안의 파일만 세션 종료 후 남습니다
OUT_DIR = f'/kaggle/working/{NAME}'
os.makedirs(OUT_DIR, exist_ok=True)

print('결과 저장 위치:', OUT_DIR)
print(f'최대 소요: 약 {EPOCHS * 10 / 60:.1f}시간 (1 epoch 약 10분 기준)')
print(f'-> {PATIENCE}회 연속 정체 시 자동 종료')
print()
print('[비교 기준] v1 최고: 에너지 19.5 / 힘 337.1 (epoch 14)')
print('[목표]      논문 MACE: EF metric 155~190')

## 5. 학습 실행

### 로그에서 볼 것

`RMSE_E_per_atom`(에너지)과 `RMSE_F`(힘) 두 숫자만 보면 됩니다.

| 지표 | v1 최고 | 목표 |
|---|---|---|
| RMSE_E_per_atom | 19.5 meV | 20 이하 유지 |
| RMSE_F | 337 meV/A | 300 아래로 |

> **`loss=` 숫자는 v1과 비교하지 마세요.** `FORCES_WEIGHT`를 100에서 1로 바꿔
> 계산식 자체가 달라졌기 때문에 약 100배 작게 표시됩니다.
> 성능이 좋아진 것이 아니므로 **비교는 반드시 RMSE로** 하세요.

정상 종료 시 아래 문구가 뜹니다.

```
Stopping optimization after 10 epochs without improvement
```

In [ ]:
loss_args = f'--loss={LOSS}'
if LOSS == 'huber':
    loss_args += f' --huber_delta={HUBER_DELTA}'

!mace_run_train \
  --name="{NAME}" \
  --train_file="{TRAIN_FILE}" \
  --valid_file="{VALID_FILE}" \
  --foundation_model="small" \
  --multiheads_finetuning=False \
  --energy_key="energy" \
  --forces_key="forces" \
  --E0s="average" \
  --batch_size={BATCH_SIZE} \
  --max_num_epochs={EPOCHS} \
  --lr={LR} \
  --patience={PATIENCE} \
  --scheduler_patience={SCHEDULER_PATIENCE} \
  {loss_args} \
  --energy_weight={ENERGY_WEIGHT} --forces_weight={FORCES_WEIGHT} \
  --ema --ema_decay=0.99 \
  --default_dtype={DTYPE} \
  --device=cuda \
  --seed=42 \
  --save_cpu \
  --keep_checkpoints \
  --save_all_checkpoints \
  --model_dir="{OUT_DIR}" \
  --checkpoints_dir="{OUT_DIR}/checkpoints" \
  --results_dir="{OUT_DIR}" \
  --log_dir="{OUT_DIR}"

## 6. 결과 정리

epoch별 오차 변화를 그래프로 보고, 최고 성적을 논문 기준과 비교합니다.

In [ ]:
import re
import matplotlib.pyplot as plt

epochs, e_rmse, f_rmse = [], [], []
pattern = re.compile(r'Epoch (\d+):.*RMSE_E_per_atom=\s*([\d.]+) meV, RMSE_F=\s*([\d.]+)')

for lf in sorted(glob.glob(f'{OUT_DIR}/*.log')):
    for line in open(lf):
        m = pattern.search(line)
        if m:
            epochs.append(int(m.group(1)))
            e_rmse.append(float(m.group(2)))
            f_rmse.append(float(m.group(3)))

if epochs:
    ef = [e + f for e, f in zip(e_rmse, f_rmse)]
    best = min(range(len(epochs)), key=lambda i: ef[i])
    print(f'총 {len(epochs)} epoch 기록')
    print(f'\n최고 성적: epoch {epochs[best]}')
    print(f'  에너지 RMSE: {e_rmse[best]:.2f} meV/atom')
    print(f'  힘 RMSE:     {f_rmse[best]:.2f} meV/A')
    print(f'  EF metric:   {ef[best]:.2f}')
    print(f'\n  v1 최고 356.6 / 논문 MACE 155~190')

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(epochs, e_rmse, marker='o', ms=3)
    ax[0].set_title('Energy RMSE (meV/atom)')
    ax[1].plot(epochs, f_rmse, marker='o', ms=3, color='tab:orange')
    ax[1].axhline(337, ls='--', c='gray', lw=1, label='v1 best (337)')
    ax[1].set_title('Force RMSE (meV/A)'); ax[1].legend()
    ax[2].plot(epochs, ef, marker='o', ms=3, color='tab:green')
    ax[2].axhline(190, ls='--', c='crimson', lw=1, label='paper MACE (190)')
    ax[2].set_title('EF metric'); ax[2].legend()
    for a in ax:
        a.set_xlabel('epoch'); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print('로그에서 epoch 기록을 찾지 못했습니다.')

## 7. 학습된 모델 내려받기

`Save & Run All`로 실행했다면, 완료 후 노트북 페이지의 **`Output`** 탭에서 파일을 받을 수 있습니다.

`.model` 파일을 맥북의 `~/mlff-project/models/` 에 넣고 `scripts/07_predict_test.py`의
`MODEL_PATH`를 그 파일로 바꾸면 제출용 CSV를 만들 수 있습니다.

In [ ]:
!ls -la "{OUT_DIR}"
print()
print('체크포인트 (매 epoch 보관):')
!ls "{OUT_DIR}/checkpoints" 2>/dev/null | head -20